# Retail Router — Demo in Microsoft Fabric
Notebook that consumes the **Azure AI Foundry** orchestrator agent (`Retail-Router-Azure-x-Snowflake`)
and displays routing (❄ Snowflake detailed / 🔭 Fabric aggregated) directly in Fabric.

## Prerequisites
1. `pip install azure-ai-projects openai azure-identity ipywidgets` (cell 1).
2. **Auth**:
   - If Fabric and Foundry are on the **same tenant** → `DefaultAzureCredential()` (notebook identity) may suffice.
   - If **different tenants** → use a **service principal from the Foundry tenant**:
     store `TENANT_ID`, `CLIENT_ID`, `CLIENT_SECRET` in a **Key Vault** and retrieve them via
     `notebookutils.credentials.getSecret(...)`, then `ClientSecretCredential(...)`.

In [ ]:
# Cell 1 — Dependencies
%pip install -q azure-ai-projects openai azure-identity ipywidgets

In [ ]:
# Cell 2 — Configuration & authentication
ENDPOINT   = "https://aistudiodemo2519002580.services.ai.azure.com/api/projects/aistudiodemo2519002580-project"
AGENT_NAME = "Retail-Router-Azure-x-Snowflake"
MODEL      = "gpt-4o"

# Auth from Fabric to Foundry (SAME tenant) using the notebook user identity.
# We retrieve a Cognitive Services token via notebookutils (reliable in Fabric),
# and wrap it in a TokenCredential for the azure-ai-projects SDK.
# The user must have the 'Cognitive Services User' / 'Foundry User' role on the account.
import time
from azure.core.credentials import AccessToken

class FabricUserCredential:
    """TokenCredential based on the Fabric user identity (notebookutils)."""
    def get_token(self, *scopes, **kwargs):
        scope = scopes[0] if scopes else "https://cognitiveservices.azure.com/.default"
        audience = scope.replace("/.default", "")
        import notebookutils
        try:
            tok = notebookutils.credentials.getToken(audience)
        except Exception:
            tok = notebookutils.credentials.getToken("https://cognitiveservices.azure.com")
        return AccessToken(tok, int(time.time()) + 3000)

try:
    cred = FabricUserCredential()
    cred.get_token("https://cognitiveservices.azure.com/.default")  # smoke test
except Exception:
    from azure.identity import DefaultAzureCredential
    cred = DefaultAzureCredential()

from azure.ai.projects import AIProjectClient
print("Auth OK — endpoint:", ENDPOINT)

In [ ]:
# Cell 3 — Agent call + routing detection
def detect_routing(resp):
    """Source cited by the agent ([Snowflake Data] / [Fabric Data]) + Cortex SQL."""
    sql = None
    for item in resp.output:
        d = item.model_dump() if hasattr(item, "model_dump") else dict(item)
        if d.get("type") == "mcp_call":
            out = d.get("output", "") or ""
            if isinstance(out, str) and "SELECT" in out.upper() and sql is None:
                up = out.upper(); i = up.find("SELECT"); j = out.find("/* Generated by Cortex */")
                sql = out[i:(j if j > i else i + 600)].replace("\\n", "\n")
    low = (resp.output_text or "").lower()
    used = []
    if "[snowflake data" in low: used.append("❄ SNOWFLAKE (detailed · RETAIL_COMPUTE_WH)")
    if "[fabric data"   in low: used.append("🔭 FABRIC (aggregated)")
    return used, sql

def ask(question):
    with AIProjectClient(endpoint=ENDPOINT, credential=cred) as pc:
        oai = pc.get_openai_client()
        try:
            resp = oai.responses.create(
                model=MODEL, input=question,
                extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
            )
            return resp.output_text, *detect_routing(resp)
        finally:
            oai.close()

# Quick test
ans, agents, sql = ask("Give me the profile of the highest-spending customer: name, city, total spent.")
print("ROUTE:", " + ".join(agents) or "(direct)")
print(ans)
if sql: print("\n--- SQL Cortex ---\n", sql)

In [ ]:
# Cell 4 — Mini chat UI (ipywidgets)
import ipywidgets as w
from IPython.display import display, Markdown, clear_output

box_q = w.Text(placeholder="Ask your retail question...", layout=w.Layout(width="80%"))
btn   = w.Button(description="Send", button_style="primary")
out   = w.Output()

SUGGEST = [
    "Give me the profile of the highest-spending customer: name, city, segment, total spent.",
    "What are the top 3 regions by revenue?",
    "What is the average basket per customer segment (Premium, Standard, Bronze)?",
    "Who is our best customer in Grenoble, and how does their region's revenue compare nationally?",
]

def on_send(_):
    q = box_q.value.strip()
    if not q: return
    with out:
        print("⏳ " + q)
        try:
            ans, agents, sql = ask(q)
            route = " + ".join(agents) if agents else "(direct response)"
            display(Markdown(f"**FOUNDRY route → {route}**\n\n{ans}"))
            if sql: display(Markdown(f"<details><summary>SQL Cortex</summary>\n\n```sql\n{sql}\n```\n</details>"))
        except Exception as e:
            print("Error:", e)
    box_q.value = ""

btn.on_click(on_send)
box_q.on_submit(on_send) if hasattr(box_q, "on_submit") else None
display(Markdown("### 🤖 Retail Router — " + " · ".join(f"`{s[:38]}...`" for s in SUGGEST[:2])))
display(w.HBox([box_q, btn]), out)